# Prompt the model

Loads one checkpoint, then continues whatever you put in `PROMPT`.

Run the setup cell once. After that, edit the prompt cell and run it again. The model stays loaded.

`MODE = "instruct"` wraps the text the way SFT was trained: `<|user|>`, your text, `<|assistant|>`, and the model writes the reply until `<|end|>`. `MODE = "raw"` continues the text exactly as written, which is how the pretrained model was asked to finish a function stub.

In [2]:
from __future__ import annotations

import sys
from pathlib import Path

import torch

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from mork.checkpoint import load_model_for_inference
from mork.evaluate import sample
from mork.tokenizer import MorkTokenizer

CHECKPOINT = ROOT / "checkpoints" / "sft" / "latest.pt"
TOKENIZER = ROOT / "data" / "tokenizer.json"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

assert CHECKPOINT.exists(), f"checkpoint not found: {CHECKPOINT}"
assert TOKENIZER.exists(), f"tokenizer not found: {TOKENIZER}"

tokenizer = MorkTokenizer.load(TOKENIZER)
model, model_cfg, step, loss = load_model_for_inference(CHECKPOINT, device=DEVICE)
print(f"checkpoint: {CHECKPOINT}")
print(f"step {step:,}  loss {loss:.4f}  device {DEVICE}")
print(
    f"L{model_cfg.n_layer} D{model_cfg.n_embd} "
    f"context {model_cfg.block_size} vocab {model_cfg.vocab_size}"
)

checkpoint: /home/mart/projects/llms/mork-2/checkpoints/sft/latest.pt
step 2,185  loss 0.5414  device cuda
L12 D768 context 512 vocab 16384


## Prompt

`TEMPERATURE = 0` is greedy, one answer. Above 0, each sample uses `SEED` plus its index. `MAX_NEW_TOKENS` is capped so the prompt plus the reply stay inside the 512-token context.

In [18]:
MODE = "instruct"  # "instruct" or "raw"
#PROMPT = "Write a Python function `count_char(s, ch)` that returns how many times ch appears in s."
PROMPT = "Write a Python function `banana(i)` that returns a string starting with 'ba' and continues with i times 'na' "

MAX_NEW_TOKENS = 160
TEMPERATURE = 0.1
TOP_K = 50
N_SAMPLES = 3
SEED = 1337


def as_model_prompt(text: str, mode: str) -> str:
    body = text.strip()
    if mode == "instruct":
        return f"<|user|>\n{body}\n<|assistant|>\n"
    if mode == "raw":
        return text
    raise ValueError(f"MODE must be 'instruct' or 'raw', got {mode!r}")


model_prompt = as_model_prompt(PROMPT, MODE)
prompt_ids = tokenizer.encode(model_prompt)
room = model_cfg.block_size - len(prompt_ids)
if room < 1:
    raise RuntimeError(
        f"prompt is {len(prompt_ids)} tokens and the context is {model_cfg.block_size}"
    )
max_new = min(MAX_NEW_TOKENS, room)
n = 1 if TEMPERATURE <= 0 else N_SAMPLES

print(f"{MODE}  {len(prompt_ids)} prompt tokens  up to {max_new} new  temperature {TEMPERATURE}")
print("--- prompt ---")
print(model_prompt)
print()

outs = sample(
    model,
    tokenizer,
    prompt_ids,
    n=n,
    max_new_tokens=max_new,
    temperature=TEMPERATURE,
    top_k=TOP_K,
    device=DEVICE,
    seed=SEED,
    stop_at_toplevel=MODE == "raw",
)
for i, text in enumerate(outs, start=1):
    reply = text.split("<|end|>", 1)[0].rstrip()
    stopped = "<|end|>" in text
    print(f"--- reply {i}{'  (stopped at <|end|>)' if stopped else ''} ---")
    print(reply)
    print()

instruct  32 prompt tokens  up to 160 new  temperature 0.1
--- prompt ---
<|user|>
Write a Python function `banana(i)` that returns a string starting with 'ba' and continues with i times 'na'
<|assistant|>


--- reply 1  (stopped at <|end|>) ---
def a_ba(i):
    if i == 0:
        return 'ba'
    else:
        return 'na' + str(i)

--- reply 2  (stopped at <|end|>) ---
def a_ba(i):
    if i == 0:
        return 'a'
    elif i == 1:
        return 'b'
    else:
        return 'ab'

--- reply 3  (stopped at <|end|>) ---
def a_ba(i):
    if i == 0:
        return "a"
    elif i == 1:
        return "b"
    else:
        return "a" + a_ba(i-1) + "b"

